In [1]:
#setup
import pandas as pd
import numpy as np
import os
import requests
import json
import prettytable
import io
import csv
import time
import glob

Running the code requires, this script to be saved under folder code, and a folder structure as follows:

	EvictionMoratoria_OD/
	|-	code      # Where the scripts are stored
	|-	data      # Data files organized as follows
	|	|-	1_raw				# All files as obtained from the source
		 |-	2_intermediate 		# All intermediate dataset

In [2]:
#list of all county FIPS codes:
geofips = pd.read_csv('..\\data\\2_intermediate\\geofips.csv')
geofips

,geofips
0,1001
1,1003
2,1005
3,1007
4,1009
...,...
3137,56037
3138,56039
3139,56041
3140,56043


In [3]:
series_ids = ["LAUCN" + str(x).zfill(5) + '0000000003' for x in geofips['geofips']]

In [4]:
series_ids

['LAUCN010010000000003',
 'LAUCN010030000000003',
 'LAUCN010050000000003',
 'LAUCN010070000000003',
 'LAUCN010090000000003',
 'LAUCN010110000000003',
 'LAUCN010130000000003',
 'LAUCN010150000000003',
 'LAUCN010170000000003',
 'LAUCN010190000000003',
 'LAUCN010210000000003',
 'LAUCN010230000000003',
 'LAUCN010250000000003',
 'LAUCN010270000000003',
 'LAUCN010290000000003',
 'LAUCN010310000000003',
 'LAUCN010330000000003',
 'LAUCN010350000000003',
 'LAUCN010370000000003',
 'LAUCN010390000000003',
 'LAUCN010410000000003',
 'LAUCN010430000000003',
 'LAUCN010450000000003',
 'LAUCN010470000000003',
 'LAUCN010490000000003',
 'LAUCN010510000000003',
 'LAUCN010530000000003',
 'LAUCN010550000000003',
 'LAUCN010570000000003',
 'LAUCN010590000000003',
 'LAUCN010610000000003',
 'LAUCN010630000000003',
 'LAUCN010650000000003',
 'LAUCN010670000000003',
 'LAUCN010690000000003',
 'LAUCN010710000000003',
 'LAUCN010730000000003',
 'LAUCN010750000000003',
 'LAUCN010770000000003',
 'LAUCN010790000000003',


In [5]:
# # Split series_ids into chunks of 50 because currently only  50 series per query are allowed
# # More info here: https://www.bls.gov/developers/api_faqs.htm#register2

# chunks = [series_ids[i:i + 50] for i in range(1, len(series_ids))]

# chunks

len(series_ids)

3142

In [6]:
chunks = [series_ids[i:i + 50] for i in range(0, len(series_ids)+1, 50)]

In [7]:
len(chunks)

63

In [11]:
api_key = '5076bf053e804fcb965cb07c041786ad'

In [12]:

url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'
api_key = '5076bf053e804fcb965cb07c041786ad'
path = '..\\'

headers = {'Content-Type': 'application/json'}

for chunk in chunks:
    data = {
        "seriesid": chunk,
        "startyear": "2018",
        "endyear": "2023",
        "registrationkey": api_key
    }
    data_json = json.dumps(data)
    
    url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'
    response = requests.post(url, data=data_json, headers=headers)
    
    json_data = json.loads(response.text)
    
    for series in json_data['Results']['series']:
        x = prettytable.PrettyTable(["series id", "year", "period", "value", "footnotes"])
        series_id = series['seriesID']
        
        for item in series['data']:
            year = item['year']
            period = item['period']
            value = item['value']
            footnotes = ",".join(footnote['text'] for footnote in item['footnotes'] if footnote)
            
            if 'M01' <= period <= 'M12':
                x.add_row([series_id, year, period, value, footnotes])
        
        output_path = f"{path}data\\2_intermediate\\unemp_bls\\{series_id}.txt"
        with open(output_path, 'w') as output:
            output.write(x.get_string())
    
    time.sleep(12)  # This line is indented to be inside the outer loop

In [13]:
# specify your path
path = '..\\data\\2_intermediate\\unemp_bls\\'

# find all .txt files in the path
all_files = glob.glob(path + "/*.txt")

# create an empty list to store dataframes
li = []

In [14]:
# loop through the list of .txt files
for filename in all_files:
    # open the file
    with open(filename, 'r') as f:
        # read the lines into a list, ignoring lines starting with '+'
        lines = [line for line in f if not line.startswith('+')]
    # join the lines back into a single string, then use StringIO to read it into a DataFrame
    df = pd.read_csv(io.StringIO(''.join(lines)), sep='|', skipfooter=1, engine='python')
    li.append(df)

In [15]:
# concatenate all dataframes in the list
df = pd.concat(li, ignore_index=True)

In [16]:
df

,Unnamed: 0,series id,year,period,value,footnotes,Unnamed: 6,series id
0,NaN,LAUCN010010000000003,2023,M12,2.2,,NaN,NaN
1,NaN,LAUCN010010000000003,2023,M11,2.2,,NaN,NaN
2,NaN,LAUCN010010000000003,2023,M10,2.3,,NaN,NaN
3,NaN,LAUCN010010000000003,2023,M09,2.3,,NaN,NaN
4,NaN,LAUCN010010000000003,2023,M08,2.5,,NaN,NaN
...,...,...,...,...,...,...,...,...
222958,NaN,LAUCN560450000000003,2018,M06,3.9,,NaN,NaN
222959,NaN,LAUCN560450000000003,2018,M05,3.3,,NaN,NaN
222960,NaN,LAUCN560450000000003,2018,M04,3.8,,NaN,NaN
222961,NaN,LAUCN560450000000003,2018,M03,4.2,,NaN,NaN


In [17]:
len(df)

222963

In [18]:
len(df.columns)

8

In [19]:
df_subset = df.iloc[:, 1:6]

In [20]:
df_subset

,series id,year,period,value,footnotes
0,LAUCN010010000000003,2023,M12,2.2,
1,LAUCN010010000000003,2023,M11,2.2,
2,LAUCN010010000000003,2023,M10,2.3,
3,LAUCN010010000000003,2023,M09,2.3,
4,LAUCN010010000000003,2023,M08,2.5,
...,...,...,...,...,...
222958,LAUCN560450000000003,2018,M06,3.9,
222959,LAUCN560450000000003,2018,M05,3.3,
222960,LAUCN560450000000003,2018,M04,3.8,
222961,LAUCN560450000000003,2018,M03,4.2,


In [21]:
df_subset.columns = ['seriesid', 'Year', 'period', 'lau_unemprate', 'footnotes']

In [22]:
df_subset

,seriesid,Year,period,lau_unemprate,footnotes
0,LAUCN010010000000003,2023,M12,2.2,
1,LAUCN010010000000003,2023,M11,2.2,
2,LAUCN010010000000003,2023,M10,2.3,
3,LAUCN010010000000003,2023,M09,2.3,
4,LAUCN010010000000003,2023,M08,2.5,
...,...,...,...,...,...
222958,LAUCN560450000000003,2018,M06,3.9,
222959,LAUCN560450000000003,2018,M05,3.3,
222960,LAUCN560450000000003,2018,M04,3.8,
222961,LAUCN560450000000003,2018,M03,4.2,


In [23]:
df_subset['geofips'] = df_subset['seriesid'].str.slice(6, 11)
df_subset['Month'] = df_subset['period'].str.slice(3, 5)

In [24]:
df_subset

,seriesid,Year,period,lau_unemprate,footnotes,geofips,Month
0,LAUCN010010000000003,2023,M12,2.2,,01001,12
1,LAUCN010010000000003,2023,M11,2.2,,01001,11
2,LAUCN010010000000003,2023,M10,2.3,,01001,10
3,LAUCN010010000000003,2023,M09,2.3,,01001,09
4,LAUCN010010000000003,2023,M08,2.5,,01001,08
...,...,...,...,...,...,...,...
222958,LAUCN560450000000003,2018,M06,3.9,,56045,06
222959,LAUCN560450000000003,2018,M05,3.3,,56045,05
222960,LAUCN560450000000003,2018,M04,3.8,,56045,04
222961,LAUCN560450000000003,2018,M03,4.2,,56045,03


In [25]:
print(df_subset['geofips'].dtypes)

object


In [26]:
df_subset.dtypes

seriesid          object
Year              object
period            object
lau_unemprate    float64
footnotes         object
geofips           object
Month             object
dtype: object

In [27]:
df_subset['seriesid'] = df_subset['seriesid'].astype(str)
df_subset['Year'] = df_subset['Year'].astype(str)
df_subset['period'] = df_subset['period'].astype(str)
df_subset['footnotes'] = df_subset['footnotes'].astype(str)
df_subset['geofips'] = df_subset['geofips'].astype(str)
df_subset['Month'] = df_subset['Month'].astype(str)

In [28]:
df_subset.dtypes

seriesid          object
Year              object
period            object
lau_unemprate    float64
footnotes         object
geofips           object
Month             object
dtype: object

In [29]:
df_subset['YearMonth'] = df_subset['Year'] + df_subset['Month']

In [30]:
df_subset

,seriesid,Year,period,lau_unemprate,footnotes,geofips,Month,YearMonth
0,LAUCN010010000000003,2023,M12,2.2,,01001,12,202312
1,LAUCN010010000000003,2023,M11,2.2,,01001,11,202311
2,LAUCN010010000000003,2023,M10,2.3,,01001,10,202310
3,LAUCN010010000000003,2023,M09,2.3,,01001,09,202309
4,LAUCN010010000000003,2023,M08,2.5,,01001,08,202308
...,...,...,...,...,...,...,...,...
222958,LAUCN560450000000003,2018,M06,3.9,,56045,06,201806
222959,LAUCN560450000000003,2018,M05,3.3,,56045,05,201805
222960,LAUCN560450000000003,2018,M04,3.8,,56045,04,201804
222961,LAUCN560450000000003,2018,M03,4.2,,56045,03,201803


In [54]:
df_subset.to_stata(path + 'county_monthly_unemp.dta')